# ***Task 1.2 — Key Assumptions***

**Paper:** Pegasos: Primal Estimated sub-GrAdient SOlver for SVM  

## *Assumption 1*

**Assumption:** The input feature vectors are bounded in L2 norm, specifically `‖x‖ ≤ R` for some constant `R`. The paper uses the normalised assumption `‖x‖ ≤ 1`.

**Why the method needs it:** The convergence bound in Theorem 1 depends explicitly on the boundedness constant `R`. The sub-gradient of the hinge loss is `yx`, so its magnitude is exactly `‖x‖`. If `‖x‖` is unbounded, a single training example could produce an arbitrarily large update to `w`, destroying the controlled convergence rate `O(R²/(λε))`.

**Violation scenario:** A bag-of-words text classification dataset where documents of vastly different lengths are used without normalisation — a 10,000-word document will have a feature vector norm hundreds of times larger than a 50-word document, making the effective R huge and the bound vacuous in practice.

**Paper reference:** Section 2, statement of Theorem 1 — `‖x‖ ≤ 1` is listed as an explicit assumption, and R appears in the final bound `O(R²/(λε))`.

## *Assumption 2*

**Assumption:** The optimal SVM weight vector `w*` lies within the L2 ball of radius `1/√λ`, i.e., `‖w*‖ ≤ 1/√λ`.

**Why the method needs it:** The projection step in Pegasos (Equation 4) projects the weight vector onto precisely this ball after every update. Without this assumption (or equivalently, without the regulariser ensuring the optimum lies in the ball), the projection step would be projecting onto an incorrect feasible set, and the guarantees of Algorithm 1 would no longer hold.

**Violation scenario:** If λ is set extremely small (e.g., `λ = 1e-10`) to allow a very large-margin solution, the ball radius `1/√λ = 10⁵` is enormous and the projection never helps tighten the iterate. In practice, choosing too small a λ causes numerical instability because the weight vector can grow without meaningful regularisation.

**Paper reference:** Section 2, Equation (4) and the accompanying text: "It is well known that the optimal solution `w*` lies in the ball of radius `1/√λ`." This is derived from the structure of the regularised primal objective.

## *Assumption 3*

**Assumption:** The algorithm assumes that a randomly sampled mini-batch provides an **unbiased stochastic estimate of the sub-gradient** of the full objective, which relies on the examples being drawn i.i.d. from the underlying distribution (or uniformly at random from the training set).

**Why the method needs it:** Pegasos's convergence proof (Theorem 1, Section 2) models the mini-batch as a random sample and uses the law of large numbers to argue that the stochastic updates decrease the expected objective value at each step. If examples are not sampled uniformly — for example, if the mini-batch is selected adversarially or in a fixed ordered sweep — the sub-gradient estimate becomes biased and the theoretical guarantee no longer applies.

**Violation scenario:** A time-series dataset where consecutive samples are highly correlated (e.g., stock prices within the same trading day) — if mini-batches are drawn as contiguous time windows, the samples within each mini-batch are not independent, the gradient estimate is biased towards the local temporal pattern, and convergence can stagnate or oscillate.

**Paper reference:** Section 3.1 (Mini-batch extension) — uniform random sampling of i.i.d. examples is explicitly assumed when computing the expected sub-gradient; the analysis references the independence of the drawn samples in the expectation argument.

## *Assumption 4*

**Assumption:** The method's primary analysis is for **linear** SVMs (inner product `⟨w, x⟩` in the original feature space). The kernel extension is mentioned but the main theory assumes the dot product is directly computable, implying that the feature map is finite-dimensional.

**Why the method needs it:** The update rule `w_{t+1/2} = (1 − 1/t)w_t + (η_t/k) Σ yx` stores `w` as an explicit d-dimensional vector. In infinite-dimensional kernel Hilbert spaces (e.g., RBF kernel), this vector cannot be stored explicitly; the kernel trick would require maintaining all training examples, which defeats the purpose of the `O(1/(λε))` memory-independent bound.

**Violation scenario:** A dataset where the decision boundary is highly non-linear (e.g., concentric rings that are not linearly separable in any finite feature expansion) — a linear Pegasos will fail to find a good classifier and no finite-dimensional feature map resolves this without an explicit kernel mapping.

**Paper reference:** Section 4 (Kernels) — the authors note that kernel Pegasos requires maintaining a growing set of support vectors and does not retain the `O(1/(λε))` iteration complexity that makes the linear version remarkable.
